# Walmart Store Sales Forecasting — Prophet Experiment
## Architecture Overview


## Setup


In [ ]:
import os, sys, warnings
import joblib
from pathlib import Path
warnings.filterwarnings('ignore')
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed
from tqdm import tqdm

from prophet import Prophet
from prophet.diagnostics import cross_validation as prophet_cv, performance_metrics
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.pipeline import Pipeline

import mlflow
import mlflow.sklearn
import dagshub

from src.wmae import wmae
from src.features import (
    load_raw_data, merge_all,
    SUPER_BOWL_DATES, LABOR_DAY_DATES, THANKSGIVING_DATES, CHRISTMAS_DATES,
    TARGET
)

SEED     = 42
DATA_DIR = PROJECT_ROOT
N_JOBS   = -1

print('Prophet version:', Prophet.__module__)
print('All imports successful')


In [ ]:
os.environ.setdefault('MLFLOW_TRACKING_USERNAME', 'sansi23')

dagshub.init(
    repo_owner='sansi23',
    repo_name='Walmart-Recruiting---Store-Sales-Forecasting',
    mlflow=True
)
prophet_experiment = mlflow.set_experiment('Prophet_Training')
print('MLflow experiment:', prophet_experiment.name)
print('Tracking URI:', mlflow.get_tracking_uri())


## Data Loading


In [ ]:
train_raw, test_raw, features_raw, stores_raw = load_raw_data(DATA_DIR)
train_raw, test_raw = merge_all(train_raw, test_raw, features_raw, stores_raw)

print(f'Train: {train_raw.shape}  |  Test: {test_raw.shape}')
print(f'Train dates: {train_raw["Date"].min()} → {train_raw["Date"].max()}')
train_raw.head()


## Exploratory Data Analysis


In [ ]:
series_lengths = train_raw.groupby(['Store','Dept']).size()
fig, ax = plt.subplots(figsize=(8, 4))
series_lengths.hist(bins=30, ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Distribution of Series Lengths (weeks per Store-Dept)')
ax.set_xlabel('Number of Weeks')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()
print(series_lengths.describe())


In [ ]:
holiday_dates_dict = {
    'SuperBowl':    SUPER_BOWL_DATES,
    'LaborDay':     LABOR_DAY_DATES,
    'Thanksgiving': THANKSGIVING_DATES,
    'Christmas':    CHRISTMAS_DATES,
}

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, (hol_name, hol_dates) in zip(axes.flat, holiday_dates_dict.items()):
    windows = []
    for hol_date in hol_dates:
        window = train_raw[
            (train_raw['Date'] >= hol_date - pd.Timedelta(weeks=4)) &
            (train_raw['Date'] <= hol_date + pd.Timedelta(weeks=4))
        ].copy()
        window['offset'] = ((window['Date'] - hol_date).dt.days / 7).astype(int)
        windows.append(window)
    if windows:
        all_w = pd.concat(windows)
        avg = all_w.groupby('offset')[TARGET].mean()
        ax.bar(avg.index, avg.values, color='tomato')
        ax.axvline(0, color='black', linestyle='--', linewidth=1.5, label='Holiday')
        ax.set_title(f'{hol_name} Effect Window')
        ax.set_xlabel('Weeks offset from holiday')
        ax.set_ylabel('Avg Weekly Sales')
        ax.legend()
plt.suptitle('Holiday Sales Windows — Average Effect', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
sample = (train_raw[(train_raw['Store']==1) & (train_raw['Dept']==1)]
          .sort_values('Date').set_index('Date')['Weekly_Sales'])

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(sample.index, sample.values, linewidth=1.5, label='Sales')
for hol_name, hol_dates, color in [
    ('Thanksgiving', THANKSGIVING_DATES, 'orange'),
    ('Christmas',    CHRISTMAS_DATES,    'red'),
    ('SuperBowl',    SUPER_BOWL_DATES,   'blue'),
]:
    for i, h in enumerate(hol_dates):
        ax.axvspan(h - pd.Timedelta(weeks=1), h + pd.Timedelta(weeks=1),
                   alpha=0.15, color=color, label=hol_name if i == 0 else '_')
ax.set_title('Store 1 Dept 1 — Sales with Holiday Windows')
ax.legend()
plt.tight_layout()
plt.show()


## Data Cleaning


In [ ]:
with mlflow.start_run(run_name='Prophet_Cleaning'):
    train_clean = train_raw.copy()
    test_clean  = test_raw.copy()

    neg_count = (train_clean[TARGET] < 0).sum()
    train_clean[TARGET] = train_clean[TARGET].clip(lower=0.0)

    for col in ['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']:
        train_clean[col] = train_clean[col].fillna(0.0).clip(lower=0.0)
        test_clean[col]  = test_clean[col].fillna(0.0).clip(lower=0.0)

    for df in [train_clean, test_clean]:
        df['MarkDown_Total'] = df[['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']].sum(axis=1)

    for col in ['CPI','Unemployment','Temperature','Fuel_Price']:
        train_clean[col] = train_clean.groupby('Store')[col].transform(lambda x: x.ffill().bfill())
        test_clean[col]  = test_clean.groupby('Store')[col].transform(lambda x: x.ffill().bfill())

    series_len    = train_clean.groupby(['Store','Dept']).size()
    valid_series  = series_len[series_len >= 26].reset_index()[['Store','Dept']]
    train_clean   = train_clean.merge(valid_series, on=['Store','Dept'])
    n_series      = len(valid_series)

    mlflow.log_params({
        'negative_sales_clipped': int(neg_count),
        'n_valid_series':         n_series,
        'train_rows':             len(train_clean),
    })
    print(f'Negatives clipped: {neg_count}')
    print(f'Valid series (≥26 weeks): {n_series}')
    print(f'Train rows: {len(train_clean)}')


## Feature Selection — Which regressors to add?


In [ ]:
with mlflow.start_run(run_name='Prophet_Feature_Selection'):
    REGRESSORS = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'MarkDown_Total']

    correlations = train_clean[REGRESSORS + [TARGET]].corr()[TARGET].drop(TARGET).abs().sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(7, 4))
    correlations.plot.bar(ax=ax, color='steelblue')
    ax.set_title('|Pearson Correlation| with Weekly Sales')
    ax.set_ylabel('|Correlation|')
    ax.axhline(0.05, color='red', linestyle='--', label='0.05 threshold')
    ax.legend()
    plt.tight_layout()
    plt.savefig('prophet_regressor_corr.png', dpi=120, bbox_inches='tight')
    plt.show()

    selected_regressors = correlations[correlations > 0.05].index.tolist()

    mlflow.log_params({
        'all_regressors':      ','.join(REGRESSORS),
        'selected_regressors': ','.join(selected_regressors),
    })
    mlflow.log_metrics({f'corr_{c}': float(correlations[c]) for c in REGRESSORS})
    mlflow.log_artifact('prophet_regressor_corr.png')

    print(f'Selected regressors: {selected_regressors}')


## Holiday DataFrame for Prophet


In [ ]:
def build_prophet_holidays():
    """Build a holiday DataFrame with ±window lower_window / upper_window for Prophet."""
    rows = []
    specs = [
        ('SuperBowl',    SUPER_BOWL_DATES,    -1, 1),
        ('LaborDay',     LABOR_DAY_DATES,     -1, 1),
        ('Thanksgiving', THANKSGIVING_DATES,  -1, 2),
        ('Christmas',    CHRISTMAS_DATES,     -2, 1),
    ]
    for name, dates, lower, upper in specs:
        for d in dates:
            rows.append({'holiday': name, 'ds': d, 'lower_window': lower, 'upper_window': upper})
    return pd.DataFrame(rows)


HOLIDAYS_DF = build_prophet_holidays()
HOLIDAYS_DF.head(10)


## Cross Validation


In [ ]:
def fit_predict_prophet(store, dept, train_df, future_df, regressors, holidays_df,
                         changepoint_prior, seasonality_prior):
    """
    Fit a Prophet model for one (Store, Dept) pair and return predictions for future_df.
    """
    series = (
        train_df[(train_df['Store'] == store) & (train_df['Dept'] == dept)]
        .sort_values('Date')
        .rename(columns={'Date': 'ds', TARGET: 'y'})
    )
    future = future_df[(future_df['Store'] == store) & (future_df['Dept'] == dept)].copy()
    if future.empty:
        return pd.DataFrame(columns=['Store', 'Dept', 'Date', 'yhat'])
    if len(series) < 10:
        future['yhat'] = series['y'].mean() if len(series) > 0 else 0.0
        return future[['Store', 'Dept', 'Date', 'yhat']]

    for reg in regressors:
        if reg not in series.columns:
            series[reg] = 0.0

    m = Prophet(
        holidays=holidays_df,
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=changepoint_prior,
        seasonality_prior_scale=seasonality_prior,
        interval_width=0.95,
    )

    for reg in regressors:
        m.add_regressor(reg)

    m.fit(series[['ds', 'y'] + regressors], iter=300)

    future_series = future.rename(columns={'Date': 'ds'}).copy()
    for reg in regressors:
        if reg not in future_series.columns:
            future_series[reg] = 0.0

    forecast = m.predict(future_series[['ds'] + regressors])
    future_series['yhat'] = forecast['yhat'].clip(lower=0).values
    future_series['Store'] = store
    future_series['Dept']  = dept
    future_series = future_series.rename(columns={'ds': 'Date'})
    return future_series[['Store', 'Dept', 'Date', 'yhat']]


In [ ]:
with mlflow.start_run(run_name='Prophet_CV'):
    VAL_WEEKS = int(test_raw['Date'].nunique())
    split_date = train_clean['Date'].max() - pd.Timedelta(weeks=VAL_WEEKS)

    tr_cv  = train_clean[train_clean['Date'] <= split_date].copy()
    val_cv = train_clean[train_clean['Date'] >  split_date].copy()

    pairs = tr_cv[['Store','Dept']].drop_duplicates().values.tolist()

    CHANGEPOINT_PRIOR  = 0.05
    SEASONALITY_PRIOR  = 10.0

    print(f'Training {len(pairs)} Prophet models in parallel...')

    cv_preds_list = Parallel(n_jobs=N_JOBS, verbose=0)(
        delayed(fit_predict_prophet)(
            store, dept, tr_cv, val_cv,
            selected_regressors, HOLIDAYS_DF,
            CHANGEPOINT_PRIOR, SEASONALITY_PRIOR
        )
        for store, dept in pairs
    )

    cv_preds = pd.concat([p for p in cv_preds_list if p is not None], ignore_index=True)
    cv_merged = val_cv.merge(cv_preds, on=['Store','Dept','Date'], how='left')
    cv_merged['yhat'] = cv_merged['yhat'].fillna(0.0)

    cv_wmae = wmae(cv_merged[TARGET].values, cv_merged['yhat'].values, cv_merged['IsHoliday'].values)

    hol_mask  = cv_merged['IsHoliday'].astype(bool)
    wmae_hol  = np.mean(np.abs(cv_merged.loc[hol_mask,  TARGET] - cv_merged.loc[hol_mask,  'yhat']))
    wmae_nhol = np.mean(np.abs(cv_merged.loc[~hol_mask, TARGET] - cv_merged.loc[~hol_mask, 'yhat']))

    mlflow.log_params({
        'changepoint_prior_scale': CHANGEPOINT_PRIOR,
        'seasonality_prior_scale': SEASONALITY_PRIOR,
        'val_weeks':               VAL_WEEKS,
        'n_validation_series':     len(pairs),
        'validation_metric':       'WMAE_raw_scale',
        'holiday_weight':          5,
        'regressors':              ','.join(selected_regressors),
    })
    mlflow.log_metrics({
        'cv_wmae':          cv_wmae,
        'cv_mae_holiday':   float(wmae_hol),
        'cv_mae_nonholiday':float(wmae_nhol),
    })

    print(f'CV WMAE: {cv_wmae:,.2f}')
    print(f'Holiday MAE: {wmae_hol:,.2f}  |  Non-Holiday MAE: {wmae_nhol:,.2f}')


In [ ]:
cv_merged['residual'] = cv_merged[TARGET] - cv_merged['yhat']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(cv_merged['residual'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Residual Distribution (CV)')
axes[0].set_xlabel('Residual')

cv_merged.boxplot(column='residual', by='IsHoliday', ax=axes[1])
axes[1].set_title('Residuals: Holiday vs Non-Holiday')
axes[1].set_xlabel('IsHoliday')
plt.tight_layout()
plt.show()


## Hyperparameter Tuning


In [15]:
with mlflow.start_run(run_name='Prophet_Tuning'):
    tuning_pairs = pairs

    regressor_candidates = [
        ('none', []),
        ('selected', selected_regressors),
        ('markdown_only', [r for r in ['MarkDown_Total'] if r in REGRESSORS]),
    ]
    regressor_sets = []
    seen_regressor_sets = set()
    for reg_name, regressors in regressor_candidates:
        reg_key = tuple(regressors)
        if reg_key not in seen_regressor_sets:
            regressor_sets.append((reg_name, regressors))
            seen_regressor_sets.add(reg_key)
    param_grid = [
        {
            'changepoint_prior_scale': cp,
            'seasonality_prior_scale': sp,
            'regressor_name': reg_name,
            'regressors': regressors,
        }
        for cp in [0.01, 0.05, 0.1]
        for sp in [1.0, 10.0]
        for reg_name, regressors in regressor_sets
    ]

    tuning_results = []

    for trial_id, params in enumerate(param_grid, start=1):
        cp = params['changepoint_prior_scale']
        sp = params['seasonality_prior_scale']
        trial_regressors = params['regressors']
        run_name = f"Prophet_trial_{trial_id:02d}_cp{cp}_sp{sp}_{params['regressor_name']}"

        with mlflow.start_run(run_name=run_name, nested=True):
            mlflow.log_params({
                **params,
                'validation_series_count': len(tuning_pairs),
                'validation_weeks': VAL_WEEKS,
                'regressors': ','.join(trial_regressors) or 'none',
                'holiday_weight': 5,
                'metric': 'WMAE_raw_scale',
            })

            preds_list = Parallel(n_jobs=N_JOBS, verbose=0)(
                delayed(fit_predict_prophet)(
                    store,
                    dept,
                    tr_cv,
                    val_cv,
                    trial_regressors,
                    HOLIDAYS_DF,
                    cp,
                    sp,
                )
                for store, dept in tuning_pairs
            )
            preds = pd.concat(
                [p for p in preds_list if p is not None],
                ignore_index=True,
            )
            merged = val_cv.merge(
                preds,
                on=['Store', 'Dept', 'Date'],
                how='left',
            )
            merged['yhat'] = merged['yhat'].fillna(0.0)
            score = wmae(
                merged[TARGET].values,
                merged['yhat'].values,
                merged['IsHoliday'].values,
            )
            holiday_mask = merged['IsHoliday'].astype(bool)
            holiday_mae = (np.mean(np.abs(merged.loc[holiday_mask, TARGET] - merged.loc[holiday_mask, 'yhat']))
                          if holiday_mask.any() else np.nan)
            nonholiday_mae = (np.mean(np.abs(merged.loc[~holiday_mask, TARGET] - merged.loc[~holiday_mask, 'yhat']))
                             if (~holiday_mask).any() else np.nan)
            mlflow.log_metrics({
                'validation_wmae': float(score),
                'validation_mae_holiday': float(holiday_mae) if np.isfinite(holiday_mae) else 0.0,
                'validation_mae_nonholiday': float(nonholiday_mae) if np.isfinite(nonholiday_mae) else 0.0,
            })

        tuning_results.append({
            'trial_id': trial_id,
            'changepoint_prior_scale': cp,
            'seasonality_prior_scale': sp,
            'regressor_name': params['regressor_name'],
            'regressors': ','.join(trial_regressors),
            'wmae': score,
        })
        print(f'trial={trial_id:02d} cp={cp} sp={sp} -> WMAE={score:,.2f}')

    tuning_df = pd.DataFrame(tuning_results).sort_values('wmae')
    best_cp = float(tuning_df.iloc[0]['changepoint_prior_scale'])
    best_sp = float(tuning_df.iloc[0]['seasonality_prior_scale'])
    best_regressor_name = tuning_df.iloc[0]['regressor_name']
    best_regressors = dict(regressor_sets)[best_regressor_name]
    best_score = float(tuning_df.iloc[0]['wmae'])

    tuning_df.to_csv('prophet_tuning_results.csv', index=False)
    mlflow.log_artifact('prophet_tuning_results.csv')
    mlflow.log_params({
        'best_changepoint_prior': best_cp,
        'best_seasonality_prior': best_sp,
        'best_regressor_set': best_regressor_name,
        'best_regressors': ','.join(best_regressors) or 'none',
        'tuning_validation_weeks': VAL_WEEKS,
    })
    mlflow.log_metric('best_tuning_wmae', best_score)

display(tuning_df)
print(f'Best: cp={best_cp} sp={best_sp} WMAE={best_score:,.2f}')


🏃 View run Prophet_trial_01_cp0.01_sp1.0_none at: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/3/runs/957536cab64440b9a1615a4df7bf5faf
🧪 View experiment at: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/3
🏃 View run Prophet_Tuning at: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/3/runs/bb74e71a91ca4ecc839bfd50e8f8006f
🧪 View experiment at: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/3


ValueError: Dataframe has no rows.

## Final Model + Pipeline + Model Registry


In [ ]:
class ProphetPipelineWrapper(BaseEstimator, RegressorMixin):
    """
    Sklearn-compatible wrapper for a collection of per-(Store,Dept) Prophet models.
    fit() trains one Prophet per series.
    predict() returns predictions aligned to the input DataFrame's (Store, Dept, Date).
    Accepts raw (unprocessed) test data.
    """
    def __init__(self, regressors=None, holidays_df=None,
                 changepoint_prior=0.05, seasonality_prior=10.0, n_jobs=-1):
        self.regressors          = regressors or []
        self.holidays_df         = holidays_df
        self.changepoint_prior   = changepoint_prior
        self.seasonality_prior   = seasonality_prior
        self.n_jobs              = n_jobs

    def _preprocess(self, df):
        """Minimal preprocessing of a raw merged DataFrame."""
        df = df.copy()
        for col in ['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']:
            df[col] = df.get(col, pd.Series(0.0, index=df.index)).fillna(0.0).clip(lower=0.0)
        df['MarkDown_Total'] = df[['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']].sum(axis=1)
        for col in ['CPI','Unemployment','Temperature','Fuel_Price']:
            if col in df.columns:
                df[col] = df.groupby('Store')[col].transform(lambda x: x.ffill().bfill())
        return df

    def fit(self, X, y=None):
        df = self._preprocess(X)
        if y is not None:
            df[TARGET] = y

        self.pairs_ = df[['Store','Dept']].drop_duplicates().values.tolist()
        self.train_df_ = df.copy()

        def _fit_one(store, dept):
            series = (df[(df['Store']==store) & (df['Dept']==dept)]
                        .sort_values('Date')
                        .rename(columns={'Date':'ds', TARGET:'y'}))
            if len(series) < 10:
                return (store, dept, None, float(series['y'].mean()) if len(series) > 0 else 0.0)
            m = Prophet(
                holidays=self.holidays_df,
                yearly_seasonality=True,
                weekly_seasonality=False,
                daily_seasonality=False,
                changepoint_prior_scale=self.changepoint_prior,
                seasonality_prior_scale=self.seasonality_prior,
            )
            for reg in self.regressors:
                m.add_regressor(reg)
            cols = ['ds','y'] + [r for r in self.regressors if r in series.columns]
            m.fit(series[cols], iter=300)
            return (store, dept, m, None)

        results = Parallel(n_jobs=self.n_jobs, verbose=0)(
            delayed(_fit_one)(s, d) for s, d in self.pairs_
        )
        self.models_ = {(s, d): (m, fallback) for s, d, m, fallback in results}
        return self

    def predict(self, X):
        df = self._preprocess(X).copy().reset_index(drop=True)
        df['_Input_Order'] = np.arange(len(df))
        all_preds = []

        for (store, dept), (m, fallback) in self.models_.items():
            sub = df[(df['Store']==store) & (df['Dept']==dept)].rename(columns={'Date':'ds'}).copy()
            if len(sub) == 0:
                continue
            if m is None:
                sub['yhat'] = fallback or 0.0
            else:
                cols = ['ds'] + [r for r in self.regressors if r in sub.columns]
                fc = m.predict(sub[cols])
                sub['yhat'] = fc['yhat'].clip(lower=0).values
            sub['Store'] = store
            sub['Dept']  = dept
            sub = sub.rename(columns={'ds':'Date'})
            all_preds.append(sub[['Store','Dept','Date','yhat']])

        preds_df = pd.concat(all_preds, ignore_index=True)
        result   = df.merge(preds_df, on=['Store','Dept','Date'], how='left')
        return result['yhat'].fillna(0.0).values


In [ ]:
FINAL_MODEL_NAME = 'WalmartSales_Prophet'

if 'best_cp' not in globals() or 'best_sp' not in globals():
    raise RuntimeError('Run Prophet_Tuning before final training.')

pipeline = Pipeline([
    ('prophet', ProphetPipelineWrapper(
        regressors=best_regressors,
        holidays_df=HOLIDAYS_DF,
        changepoint_prior=best_cp,
        seasonality_prior=best_sp,
        n_jobs=N_JOBS,
    ))
])

print('Fitting selected Prophet pipeline on all training rows.')
pipeline.fit(train_clean, train_clean[TARGET].values)

pipeline_predictions = np.asarray(pipeline.predict(test_clean), dtype=float)
assert len(pipeline_predictions) == len(test_clean)
assert np.isfinite(pipeline_predictions).all()
assert (pipeline_predictions >= 0).all()
PROPHET_PIPELINE_PATH = 'prophet_pipeline.joblib'
joblib.dump(pipeline, PROPHET_PIPELINE_PATH)

with mlflow.start_run(run_name='Prophet_Final') as run:
    mlflow.log_params({
        'model_family': 'Prophet',
        'registered_model_name': FINAL_MODEL_NAME,
        'selection_source': 'Prophet_Tuning',
        'final_changepoint_prior': best_cp,
        'final_seasonality_prior': best_sp,
        'regressors': ','.join(best_regressors) or 'none',
        'final_train_rows': len(train_clean),
        'test_rows': len(test_clean),
        'validation_weeks': VAL_WEEKS,
        'feature_contract': 'raw_merged_to_weekly_sales',
        'preserves_input_order': True,
    })
    mlflow.log_metric('selection_wmae', float(best_score))
    if 'cv_wmae' in globals():
        mlflow.log_metric('reference_cv_wmae', float(cv_wmae))
    mlflow.set_tags({
        'pipeline_contract': 'raw_merged_dataframe_to_weekly_sales',
        'winner_selected_by': 'minimum_validation_wmae',
    })
    mlflow.log_artifact(PROPHET_PIPELINE_PATH)
    mlflow.sklearn.log_model(
        pipeline,
        name='prophet_pipeline',
        registered_model_name=FINAL_MODEL_NAME,
        serialization_format='cloudpickle',
    )

print(f'Selected validation WMAE: {best_score:,.2f}')
print(f'Run ID: {run.info.run_id}')
print(f'Model registered as: {FINAL_MODEL_NAME}')
